# <u> NOTEBOOK PERMETTANT DE REPONDRE AUX DEMANDES D'ANNABELLE </U>

### <u> Import des librairies utiles pour ce projet </u>

In [523]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go
import statsmodels.api as sm

In [524]:
# Palette de couleurs
px.colors.qualitative.LibrairieContrast = [
    "#2f3a4a",  
    "#7f5083",  
    "#e95c6d",  
    "#ffa600",  
    "#4c9f70", 
    "#3b7dd8", 
    "#b95383"   
]

px.colors.qualitative.Librairie3 = [
    "#e95c6d",
    "#2f3a4a",
    "#ffa600",
]

palette = {
    "0": "#e95c6d",
    "1": "#2f3a4a",
    "2": "#ffa600"
}



### <u> Chargement du dataset selon les csv remis lors de la mission </u>

In [525]:
df_lapage=pd.read_csv(r'..\..\projet9_Lapage\data\processed\df_lapage.csv',sep=';', encoding='utf-8')

In [526]:
print('Les données minimales par colonnes du dataframe sont les suivantes :')
print(df_lapage.min())

print('\n ----------------------------------------------------------')
print('Les données maximales par colonnes du dataframe sont les suivantes :')
print(df_lapage.max())

print('\n ----------------------------------------------------------')
print('Le type de chaque colonne du dataframe est :')
print(df_lapage.dtypes)

Les données minimales par colonnes du dataframe sont les suivantes :
id_prod              0_0
date          2021-03-01
session_id           s_1
client_id            c_1
price               0.62
categ                  0
sex                    f
birth               1929
dtype: object

 ----------------------------------------------------------
Les données maximales par colonnes du dataframe sont les suivantes :
id_prod             2_99
date          2023-02-28
session_id       s_99999
client_id          c_999
price              300.0
categ                  2
sex                    m
birth               2004
dtype: object

 ----------------------------------------------------------
Le type de chaque colonne du dataframe est :
id_prod        object
date           object
session_id     object
client_id      object
price         float64
categ           int64
sex            object
birth           int64
dtype: object


In [527]:
# Conversion de la colonne date en datetime
df_lapage['date'] = pd.to_datetime(df_lapage['date'])

## <u> 1. Le chiffre d'affaires </U>

#### Chiffre d’affaires avec la moyenne mobile : jour, semaine, mois

In [528]:
# Agregation en jour pour ensuite effectuer la moyenne glissante en semaine 
df_ca = df_lapage.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().to_frame().reset_index()

In [529]:
# Création de la colonne indiquant une moyenne glissante sur une semaine
df_ca['Moyenne glissante (semaine)']=df_ca['price'].rolling(window=7).mean().round(2)

In [530]:
# Création de la colonne indiquant une moyenne glissante sur un mois
df_ca['Moyenne glissante (mois)']=df_ca['price'].rolling(window=30).mean().round(2)

In [531]:
# Vérification de la création des colonnes de moyennes glissantes
df_ca.head(35)

,date,price,Moyenne glissante (semaine),Moyenne glissante (mois)
0,2021-03-01,16565.22,NaN,NaN
1,2021-03-02,15486.45,NaN,NaN
2,2021-03-03,15198.69,NaN,NaN
3,2021-03-04,15196.07,NaN,NaN
4,2021-03-05,17471.37,NaN,NaN
5,2021-03-06,15785.28,NaN,NaN
6,2021-03-07,14760.20,15780.47,NaN
7,2021-03-08,15679.53,15653.94,NaN
8,2021-03-09,15710.51,15685.95,NaN
9,2021-03-10,15496.87,15728.55,NaN


In [532]:
# Grahique concernant la saisonnalité du CA. 
# Pas de réelle saisonnalité
# Interprétation limitée et tendance à une certaine stabilité - saisonnalité marquée mais pas de pics

figure = px.line(
    df_ca, 
    x='date', 
    y=['price', 'Moyenne glissante (semaine)','Moyenne glissante (mois)'], 
    color_discrete_sequence=list(palette.values()),
    markers=False,
    title='Évolution du chiffre d\'affaires',
    labels={
        'date': 'Date (mois)',
        'value': 'Chiffre d’affaires (€)',
        'variable': 'Moyennes glissantes'
    }
)

figure.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure.update_traces(
    hovertemplate='%{y:.0f}€'
)

figure.show()

In [533]:
# Quanti - quanti



#### Chiffre d’affaires par catégorie + évolution dans le temps

In [534]:
# Création d'un df contenant les données agregées par mois
df_lapage_mois = df_lapage.groupby([pd.Grouper(key='date', freq='MS'), 'categ'])['price'].sum().reset_index()
df_lapage_mois['date AAAA-MM'] = df_lapage_mois['date'].dt.strftime('%Y-%m')

In [535]:
# Vérification des catégories uniques présentes dans la colonne categ
categories =df_lapage_mois.categ.unique()
print('Les catégories uniques du df sont les suivantes :', categories)

Les catégories uniques du df sont les suivantes : [0 1 2]


In [536]:
# Affichage du chiffre d'affaire par catégorie sur l'intégralité de la période du dataset
for categ in df_lapage_mois.categ.unique():
    ca = df_lapage_mois[df_lapage_mois['categ'] == categ]['price'].sum()
    print (f'Le chiffre d\'affaire total de la catégorie {categ} est de {ca:,.0f} euros'.replace(',',' '))

Le chiffre d'affaire total de la catégorie 0 est de 4 419 731 euros
Le chiffre d'affaire total de la catégorie 1 est de 4 827 657 euros
Le chiffre d'affaire total de la catégorie 2 est de 2 780 275 euros


In [537]:
figure2 = px.line(
    df_lapage_mois, 
    x='date', 
    y='price', 
    color='categ',
    color_discrete_sequence=list(palette.values()),
    markers=False,
    title='Évolution mensuelle du chiffre d’affaires par catégorie',
    labels={
        'date': 'Date (mois)',
        'price': 'Chiffre d’affaires (€)',
        'categ': 'Catégories'
    }
)

figure2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure2.update_traces(
    hovertemplate='%{y:.0f}€'
    )


figure2.show()

## <u> 2. Les clients </U>

#### Nombre de clients par mois + évolution dans le temps

In [538]:
#Affichage du nombre de clients au total sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['client_id'].nunique()} clients distincts.')

Sur l'intégralité de la période de référence du dataset, il y a eu un total de 8600 clients distincts.


In [539]:
#Affichage du nombre de clients uniques par mois
clients_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].nunique().reset_index(name='Nombre de clients'))
clients_par_mois.head()

,date,Nombre de clients
0,2021-03-01,5676
1,2021-04-01,5674
2,2021-05-01,5644
3,2021-06-01,5659
4,2021-07-01,5672


In [540]:
figure3=px.line(
    clients_par_mois,
    x='date',
    y='Nombre de clients',
    color_discrete_sequence=list(palette.values()),
    markers=True,
    title='Évolution mensuelle du nombre de clients (uniques)',
    labels={
        'date': 'Date (mois)'
    }
)

figure3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure3.show()

In [541]:
#Affichage du nombre de clients par mois
clients_total_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].count().reset_index(name='Nombre de clients'))
clients_total_par_mois.head()

,date,Nombre de clients
0,2021-03-01,28601
1,2021-04-01,28443
2,2021-05-01,28285
3,2021-06-01,26850
4,2021-07-01,24738


In [542]:
figure4=px.line(
    clients_total_par_mois,
    x='date',
    y='Nombre de clients',
    color_discrete_sequence=list(palette.values()),
    markers=True,
    title='Évolution mensuelle du nombre de clients',
    labels={
        'date': 'Date (mois)'
    }
)

figure4.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure4.show()

## <u> 3. Les transactions </U>

#### Nombre de transactions + évolution dans le temps

In [543]:
#Affichage du nombre de transactions totales sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['session_id'].count()} transactions.')

Sur l'intégralité de la période de référence du dataset, il y a eu un total de 687534 transactions.


In [544]:
#Affichage du nombre de transactions par semaine

# Création d'un dataframe servant de base quotidienne
df_quotidien = df_lapage.groupby(pd.Grouper(key='date', freq='D'))['session_id'].count().reset_index(name='Transactions_jour')

# Puis création des colonnes de tendances 
df_quotidien['Tendance_Semaine'] = df_quotidien['Transactions_jour'].rolling(window=7).mean()
df_quotidien['Tendance_Mois'] = df_quotidien['Transactions_jour'].rolling(window=30, center=True).mean()
df_quotidien['Tendance_Trimestre'] = df_quotidien['Transactions_jour'].rolling(window=90, center=True).mean()
# center=True calcule la moyenne en prenant les jours avant et après le point central plutôt que la moyenne des X jours précédents.
### Prendre en compte le fait que la période commence le 01/03/21 donc le premier trimestre n'est pas significatif
### Le dernier trimestre de la période du dataset ne sera pas significatif non plus car il ne couvre que janvier et février 2023

In [545]:
figure5=px.line(
    df_quotidien,
    x='date',
    y=['Tendance_Semaine', 'Tendance_Mois', 'Tendance_Trimestre'],    
    color_discrete_sequence=list(palette.values()),
    markers=False,
    title='Évolution du nombre de transactions par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure5.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)


## <u> 4. Les produits </U>

#### Nombre de produits vendus + évolution dans le temps

In [546]:
# tous les produits sur toute la période, puis par jour, mois, par semaine, par trimestre
#Affichage du nombre de produits vendus par jour
df_prod = (df_lapage.groupby(pd.Grouper(key='date', freq='D'))['id_prod'].nunique().reset_index(name='Tendance_Prod_Jour'))
df_prod

,date,Tendance_Prod_Jour
0,2021-03-01,646
1,2021-03-02,637
2,2021-03-03,643
3,2021-03-04,616
4,2021-03-05,620
...,...,...
725,2023-02-24,610
726,2023-02-25,575
727,2023-02-26,590
728,2023-02-27,711


In [547]:
#Affichage du nombre de produits vendus par semaine
df_prod['Tendance_Prod_Semaine'] = df_prod['Tendance_Prod_Jour'].rolling(window=7, center=True).mean()

In [548]:
#Affichage du nombre de produits vendus par semaine
df_prod['Tendance_Prod_Mois'] = df_prod['Tendance_Prod_Jour'].rolling(window=30, center=True).mean()

In [549]:
#Affichage du nombre de produits vendus par trimestre
df_prod['Tendance_Prod_Trimestre'] = df_prod['Tendance_Prod_Jour'].rolling(window=90, center=True).mean()

In [550]:
figure6=px.line(
    df_prod,
    x='date',
    y=['Tendance_Prod_Semaine','Tendance_Prod_Mois','Tendance_Prod_Trimestre'],    
    color_discrete_sequence=list(palette.values()),
    markers=False,
    title='Évolution du nombre de produits uniques vendus par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure6.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure6.update_traces(
    hovertemplate='%{y:.0f}€'
    )

figure6.show()

### <u> Top, Flop, la répartition par catégorie</u>

### <u> ***Le Top 10*** </u> 

In [551]:
# Création d'une colonne dans le df principal, indiquant la quantité de produits vendus, par article
df_lapage['quantity'] = df_lapage.groupby('id_prod')['id_prod'].transform('count')
df_lapage.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity
0,0_1259,2021-03-01,s_1,c_329,11.99,0,f,1967,341
1,0_1390,2021-03-01,s_2,c_664,19.37,0,m,1960,880
2,0_1352,2021-03-01,s_3,c_580,4.50,0,m,1988,1004
3,0_1458,2021-03-01,s_4,c_7912,6.55,0,f,1989,1065
4,0_1358,2021-03-01,s_5,c_2033,16.49,0,f,1956,1106


In [552]:
# Création d'un dataframe restreint contenant la colonne quantité
df_quantity = df_lapage[['id_prod', 'price', 'categ', 'quantity']]
df_quantity.head()

,id_prod,price,categ,quantity
0,0_1259,11.99,0,341
1,0_1390,19.37,0,880
2,0_1352,4.50,0,1004
3,0_1458,6.55,0,1065
4,0_1358,16.49,0,1106


In [553]:
# Eviter le warning concernant les bugs de pandas sur les copies de dataframes
# df_quantity = df_quantity.copy()

df_quantity['ca_par_produit'] = df_quantity['quantity'] * df_quantity['price']

df_quantity = (df_quantity.groupby(['id_prod', 'categ'], as_index=False).agg({
        'quantity': 'sum',
        'price': 'mean',
        'ca_par_produit': 'sum'
    })
)
df_quantity

C:\Users\julie\AppData\Local\Temp\ipykernel_11608\2215583699.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_quantity['ca_par_produit'] = df_quantity['quantity'] * df_quantity['price']


,id_prod,categ,quantity,price,ca_par_produit
0,0_0,0,1542564,3.75,5784615.00
1,0_1,0,237169,10.99,2606487.31
2,0_10,0,484,17.95,8687.80
3,0_100,0,9,20.60,185.40
4,0_1000,0,186624,6.84,1276508.16
...,...,...,...,...,...
3260,2_95,2,16,98.99,1583.84
3261,2_96,2,357604,47.91,17132807.64
3262,2_97,2,169,160.99,27207.31
3263,2_98,2,1,149.74,149.74


In [554]:
# Les tops par catégorie
tops = [] 

for categ in df_quantity['categ'].unique():
    top10=(df_quantity[df_quantity['categ'] == categ].sort_values('ca_par_produit', ascending=False).head(10))
    tops.append(top10)
    print('\n--------------------------------------------------------------------------------------------- ')
    print(f'Le top 10 des produits les plus vendus de la catégorie {categ} est le suivant :')
    print('---------------------------------------------------------------------------------------------\n')
    print(top10[['id_prod', 'categ','quantity', 'price']])
df_top10 = pd.concat(tops)
df_top10["categ"] = df_top10["categ"].astype(str)


--------------------------------------------------------------------------------------------- 
Le top 10 des produits les plus vendus de la catégorie 0 est le suivant :
---------------------------------------------------------------------------------------------

    id_prod  categ  quantity  price
487  0_1441      0   1525225  18.99
465  0_1421      0   1324801  19.99
457  0_1414      0   1322500  19.38
460  0_1417      0   1411344  17.99
475  0_1430      0   1490841  16.47
477  0_1432      0   1572516  15.36
498  0_1451      0   1177225  19.99
494  0_1448      0   1194649  18.94
525  0_1476      0   1390041  15.66
522  0_1473      0   1338649  15.99

--------------------------------------------------------------------------------------------- 
Le top 10 des produits les plus vendus de la catégorie 1 est le suivant :
---------------------------------------------------------------------------------------------

     id_prod  categ  quantity  price
2591   1_369      1   5475600  23.99


In [560]:
# Subplor : 1 ligne, 3 colonnes
fig = sp.make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"Catégorie {c}" for c in sorted(df_top10['categ'].unique())],
    shared_yaxes=False  # échelles indépendantes
)

# Itération pourafficher un grapgique par catégorie
for i, categ in enumerate(sorted(df_top10['categ'].unique()), start=1):
    df_cat = df_top10[df_top10['categ'] == categ]

    fig.add_trace(
        go.Bar(
            x=df_cat["id_prod"],
            y=df_cat["ca_par_produit"],
            name=f"Catégorie {categ}",
            marker_color=palette[categ]
        ),
        row=1, col=i
    )

fig.update_layout(
    height=450,
    width=1200,
    title="Top 10 des produits par catégorie",
    title_x=0.5,
    showlegend=False,  
    plot_bgcolor="white",
    font=dict(size=12)
)

fig.update_yaxes(title="CA par produit (€)")

fig.show()

### <u> ***Le Flop 10*** </u> 

In [556]:
# Les flops par catégorie
flops = [] 

for categ in df_quantity['categ'].unique():
    flop10=(df_quantity[df_quantity['categ'] == categ].sort_values('ca_par_produit', ascending=True).head(10))
    flops.append(flop10)
    print('\n--------------------------------------------------------------------------------------------- ')
    print(f'Le flop 10 des produits les moins vendus de la catégorie {categ} est le suivant :')
    print('---------------------------------------------------------------------------------------------\n')
    print(flop10[['id_prod', 'categ','quantity', 'price']])
df_flop10 = pd.concat(flops)
df_flop10["categ"] = df_flop10["categ"].astype(str)



--------------------------------------------------------------------------------------------- 
Le flop 10 des produits les moins vendus de la catégorie 0 est le suivant :
---------------------------------------------------------------------------------------------

     id_prod  categ  quantity  price
595   0_1539      0         1   0.99
313   0_1284      0         1   1.38
665   0_1601      0         1   1.99
2079   0_807      0         1   1.99
1784   0_541      0         1   1.99
802   0_1728      0         1   2.27
549   0_1498      0         1   2.48
2108   0_833      0         1   2.99
166   0_1151      0         1   2.99
1792   0_549      0         1   2.99

--------------------------------------------------------------------------------------------- 
Le flop 10 des produits les moins vendus de la catégorie 1 est le suivant :
---------------------------------------------------------------------------------------------

     id_prod  categ  quantity  price
2648   1_420      1   

In [561]:
# Subplor : 1 ligne, 3 colonnes
fig = sp.make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"Catégorie {c}" for c in sorted(df_flop10['categ'].unique())],
    shared_yaxes=False  # échelles indépendantes
)

# Itération pourafficher un grapgique par catégorie
for i, categ in enumerate(sorted(df_flop10['categ'].unique()), start=1):
    df_cat = df_flop10[df_flop10['categ'] == categ]

    fig.add_trace(
        go.Bar(
            x=df_cat["id_prod"],
            y=df_cat["ca_par_produit"],
            name=f"Catégorie {categ}",
            marker_color=palette[categ]
        ),
        row=1, col=i
    )

fig.update_layout(
    height=450,
    width=1200,
    title="Top 10 des produits par catégorie",
    title_x=0.5,
    showlegend=False,  
    plot_bgcolor="white",
    font=dict(size=12)
)

fig.update_yaxes(title="CA par produit (€)")

fig.show()


## <u> 5. Le profil de nos clients </U>

### <u> SEGMENTATION DES CLIENTS : BtoB ET BtoC </u>

#### ***Il existe plusieurs moyens pour identifier les B to B :*** 

1. Classification selon le montant du CA total :    

In [558]:
# Nous identifions 4 clients ayant généré plusieurs centaines de milliers d'euros de CA

ca_par_client = df_lapage.groupby(['client_id'])['price'].sum().reset_index().sort_values('price', ascending=False)
ca_par_client.head(10)
# clients = ['c_1609','c_4958','c_6714','c_3454']

,client_id,price
677,c_1609,326039.89
4388,c_4958,290227.03
6337,c_6714,153918.60
2724,c_3454,114110.57
634,c_1570,5285.82
2513,c_3263,5276.87
1268,c_2140,5260.18
2108,c_2899,5214.05
7006,c_7319,5155.77
7715,c_7959,5135.75


In [562]:
# Création d'une colonne dans le df, précisant s'il s'agit d'un client B2B ou B2C selon le tri ci-dessus
clients = ['c_1609','c_4958','c_6714','c_3454']
df_lapage['type_client']=df_lapage['client_id'].apply(lambda x: 'B to B' if x in clients else 'B to C')
df_lapage.head(20)

# Création d'un df b2b 
df_b2b = df_lapage[df_lapage['client_id'].isin(clients)]

# Mise à jour du df principal excluant ces 4 clients
df_b2c = df_lapage[df_lapage['type_client'] == 'B to C']
df_b2c

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
0,0_1259,2021-03-01,s_1,c_329,11.99,0,f,1967,341,B to C
1,0_1390,2021-03-01,s_2,c_664,19.37,0,m,1960,880,B to C
2,0_1352,2021-03-01,s_3,c_580,4.50,0,m,1988,1004,B to C
3,0_1458,2021-03-01,s_4,c_7912,6.55,0,f,1989,1065,B to C
4,0_1358,2021-03-01,s_5,c_2033,16.49,0,f,1956,1106,B to C
...,...,...,...,...,...,...,...,...,...,...
687529,1_508,2023-02-28,s_348444,c_3573,21.92,1,f,1996,275,B to C
687530,2_37,2023-02-28,s_348445,c_50,48.99,2,f,1994,882,B to C
687531,1_695,2023-02-28,s_348446,c_488,26.99,1,f,1985,405,B to C
687532,0_1547,2023-02-28,s_348447,c_4848,8.99,0,m,1953,568,B to C


In [563]:
# répartition par catégorie
nb_categorie=df_b2c.groupby('categ')['id_prod'].nunique().reset_index()
print(nb_categorie)


   categ  id_prod
0      0     2290
1      1      737
2      2      235


2. Classification selon le nombre d'exemplaires achetés, pour chaque produit :

In [ ]:
df_lapage['quantity'] = (df_lapage.groupby(['id_prod', 'date', 'client_id'])['id_prod'].transform('count'))
df_lapage=df_lapage.sort_values('quantity',ascending=False)
df_lapage.loc[df_lapage['quantity'] > 1].head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
342807,0_1630,2022-02-26,s_171121,c_1609,12.48,0,m,1980,3,B to B
86967,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to C
126788,2_102,2021-07-18,s_64148,c_4958,59.14,2,m,1999,3,B to B
126321,2_102,2021-07-18,s_63848,c_4958,59.14,2,m,1999,3,B to B
86950,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to C


In [ ]:
df_b2b

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
6,0_1304,2021-03-01,s_7,c_1609,5.86,0,m,1980,324,B to B
13,0_1159,2021-03-01,s_7,c_1609,7.99,0,m,1980,485,B to B
50,0_1431,2021-03-01,s_33,c_3454,10.99,0,m,1969,1282,B to B
78,0_1425,2021-03-01,s_46,c_1609,12.99,0,m,1980,1266,B to B
88,0_1469,2021-03-01,s_53,c_1609,14.99,0,m,1980,1026,B to B
...,...,...,...,...,...,...,...,...,...,...
687482,0_1333,2023-02-28,s_348416,c_6714,5.99,0,f,1968,959,B to B
687494,1_392,2023-02-28,s_348416,c_6714,18.11,1,f,1968,1899,B to B
687496,2_13,2023-02-28,s_348403,c_4958,50.99,2,m,1999,350,B to B
687498,1_183,2023-02-28,s_348416,c_6714,24.99,1,f,1968,202,B to B


In [ ]:
figureb2b = px.histogram(
    df_b2b,
    x="client_id",
    color="categ",
    barmode="group",
    nbins=20,
    title="Histogramme représentant le nombre d'achats par clients B to B, par catégories"
)

figureb2b.show()

In [ ]:
var_a = df_b2b.groupby(['client_id','categ'])['price'].sum().reset_index()
df_b2b

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
6,0_1304,2021-03-01,s_7,c_1609,5.86,0,m,1980,324,B to B
13,0_1159,2021-03-01,s_7,c_1609,7.99,0,m,1980,485,B to B
50,0_1431,2021-03-01,s_33,c_3454,10.99,0,m,1969,1282,B to B
78,0_1425,2021-03-01,s_46,c_1609,12.99,0,m,1980,1266,B to B
88,0_1469,2021-03-01,s_53,c_1609,14.99,0,m,1980,1026,B to B
...,...,...,...,...,...,...,...,...,...,...
687482,0_1333,2023-02-28,s_348416,c_6714,5.99,0,f,1968,959,B to B
687494,1_392,2023-02-28,s_348416,c_6714,18.11,1,f,1968,1899,B to B
687496,2_13,2023-02-28,s_348403,c_4958,50.99,2,m,1999,350,B to B
687498,1_183,2023-02-28,s_348416,c_6714,24.99,1,f,1968,202,B to B


In [ ]:
figureb2b_ca = px.histogram(
    df_b2b,
    x="client_id",
    y="price",
    barmode="group",
    nbins=20,
    title="Histogramme représentant le montant du CA par clients B to B, par catégories"
)

figureb2b_ca.show()

 ***Le fait de trier les B2B sur la quantité n'est pas cohérent car une même personne peut acheter plusieurs exemplaires d'un même ouvrage*** 
 ***(sortie de livres, achat pour offrir ...) : 425 clients***

### <u> Utilisation du df_b2c pour identifier le profil des clients </u> 

In [ ]:
# classement de l'Age des clients par catégorie
df_b2c.sort_values(by=['categ', 'birth'], ascending=[True, True]).reset_index(drop=True)

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
0,0_1563,2021-03-25,s_11322,c_8362,3.99,0,f,1929,698,B to C
1,0_1159,2021-04-07,s_17164,c_577,7.99,0,m,1929,485,B to C
2,0_1729,2021-04-15,s_20794,c_577,14.99,0,m,1929,114,B to C
3,0_655,2021-04-15,s_20794,c_577,9.99,0,m,1929,61,B to C
4,0_1626,2021-05-02,s_28633,c_577,14.02,0,m,1929,763,B to C
...,...,...,...,...,...,...,...,...,...,...
640729,2_39,2023-02-28,s_348194,c_7483,57.99,2,m,2004,915,B to C
640730,2_208,2023-02-28,s_348339,c_6464,54.87,2,m,2004,831,B to C
640731,2_155,2023-02-28,s_348357,c_1852,46.99,2,m,2004,615,B to C
640732,2_163,2023-02-28,s_348364,c_3009,68.99,2,m,2004,559,B to C


In [ ]:
df_b2c.info()
df_b2c.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 640734 entries, 0 to 687533
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   id_prod      640734 non-null  object        
 1   date         640734 non-null  datetime64[ns]
 2   session_id   640734 non-null  object        
 3   client_id    640734 non-null  object        
 4   price        640734 non-null  float64       
 5   categ        640734 non-null  int64         
 6   sex          640734 non-null  object        
 7   birth        640734 non-null  int64         
 8   quantity     640734 non-null  int64         
 9   type_client  640734 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(3), object(5)
memory usage: 53.8+ MB


,date,price,categ,birth,quantity
count,640734,640734.000000,640734.000000,640734.000000,640734.000000
mean,2022-03-01 07:53:09.800448,17.391565,0.446833,1977.792024,753.040530
min,2021-03-01 00:00:00,0.620000,0.000000,1929.000000,1.000000
25%,2021-09-10 00:00:00,8.990000,0.000000,1971.000000,316.000000
50%,2022-02-27 00:00:00,13.990000,0.000000,1979.000000,679.000000
75%,2022-08-28 00:00:00,19.040000,1.000000,1987.000000,1055.000000
max,2023-02-28 00:00:00,300.000000,2.000000,2004.000000,2340.000000
std,NaN,18.003423,0.591360,13.879682,536.638858


In [ ]:
# Création de la colonne age, correspondant à la différence entre la date du jour et l'année de 'birth' 
# Pour que la bonne catégorie d'âge soit retenue, il faut prendre la date d'achat comme date de référence

df_b2c['date'] = pd.to_datetime(df_b2c['date'])
df_b2c['birth'] = pd.to_datetime(df_b2c['birth'], format='%Y')
df_b2c['age'] = (df_b2c['date'] - df_b2c['birth']).dt.days // 365

In [ ]:
df_b2c

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client,age
0,0_1259,2021-03-01,s_1,c_329,11.99,0,f,1967-01-01,341,B to C,54
1,0_1390,2021-03-01,s_2,c_664,19.37,0,m,1960-01-01,880,B to C,61
2,0_1352,2021-03-01,s_3,c_580,4.50,0,m,1988-01-01,1004,B to C,33
3,0_1458,2021-03-01,s_4,c_7912,6.55,0,f,1989-01-01,1065,B to C,32
4,0_1358,2021-03-01,s_5,c_2033,16.49,0,f,1956-01-01,1106,B to C,65
...,...,...,...,...,...,...,...,...,...,...,...
687529,1_508,2023-02-28,s_348444,c_3573,21.92,1,f,1996-01-01,275,B to C,27
687530,2_37,2023-02-28,s_348445,c_50,48.99,2,f,1994-01-01,882,B to C,29
687531,1_695,2023-02-28,s_348446,c_488,26.99,1,f,1985-01-01,405,B to C,38
687532,0_1547,2023-02-28,s_348447,c_4848,8.99,0,m,1953-01-01,568,B to C,70


In [ ]:
figure6 = px.histogram(
    df_b2c,
    x="age",
    color="categ",
    barmode="group",
    nbins=20,
    title="Histogramme empilé des âges par catégorie"
)

figure6.show()


In [ ]:
# Top 10 des clients en terme de CA généré
df_b2c['ca_par_client']=df_b2c['price']*df_b2c['quantity']

df_ca_clients=(
    df_b2c
    .groupby('client_id', as_index=False).agg({'ca_par_client': 'sum'})
    .rename(columns={'ca_par_client': 'ca_total'})
    .sort_values('ca_total', ascending=False)
)

df_clients_top10=df_ca_clients.head(10)

In [ ]:
df_ca_clients.describe()

,ca_total
count,8.596000e+03
mean,9.547501e+05
std,7.149360e+05
min,1.269200e+03
25%,4.158160e+05
50%,7.695504e+05
75%,1.310901e+06
max,4.468346e+06


In [ ]:
# Graphique du Top 10 des clients en terme de CA généré
figure9=px.bar(
    df_clients_top10,
    x="ca_total",
    y="client_id",
    # color="categ",
    barmode="group",
    orientation='h',
    title="Histogramme du Top10 des clients en terme de CA généré"
)
figure9.update_layout(
    yaxis={'categoryorder':'total ascending'}, 
    xaxis_title="CA total",
    yaxis_title="Client ID",
)

figure9.show()

In [ ]:
# Flop 20 des clients en terme de CA généré
df_clients_flop20=df_ca_clients.tail(20)

In [ ]:
# Graphique du Flop 20 des clients en terme de CA généré
figure10=px.bar(
    df_clients_flop20,
    x="ca_total",
    y="client_id",
    # color="categ",
    barmode="group",
    orientation='h',
    title="Histogramme du Flop20 des clients en terme de CA généré"
)
figure10.update_layout(
    yaxis={'categoryorder':'total descending'}, 
    xaxis_title="CA total",
    yaxis_title="Client ID",
)

figure10.show()

In [ ]:
# Courbe de Lorenz
df_clients = (df_ca_clients.groupby('client_id', as_index=False).agg({'ca_total': 'sum'}))

In [ ]:

b = df_clients[df_clients['col'] < 0]
a = -b['col2'].values
n = len(a)
lorenz = np.cumsum(np.sort(a)) / a.sum()
# La courbe de Lorenz commence à 0
lorenz = np.append([0],lorenz) 

# Il y a un segment de taille n pour chaque individu, plus 1 segment supplémentaire d'ordonnée 0. 
# Le premier segment commence à 0-1/n, et le dernier termine à 1+1/n.

xaxis = np.linspace(0-1/n,1+1/n,n+1)
plt.plot(xaxis,lorenz,drawstyle='steps-post')
plt.show()

KeyError: 'col'